# Yambda Brief Profile

Short exploratory notebook for the Yambda interaction parquet currently present in `data/raw/`.

Targets:
- `data/raw/yambda_interactions.parquet`
- optional side input: `data/raw/yambda_item_embeddings.parquet`

This notebook includes:
- fast Polars summaries for scale and data quality
- event, temporal, and activity profiling
- an optional `ydata-profiling` HTML report when that package is installed


In [10]:
import os
from pathlib import Path

import pandas as pd
import polars as pl

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from ydata_profiling import ProfileReport
except ImportError:
    ProfileReport = None

pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(12)

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from the current working directory")

def find_data_root(repo_root: Path) -> Path:
    override = os.environ.get("RECSYS_GEN_DATA_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    return repo_root / "data"

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_ROOT = find_data_root(REPO_ROOT)
INTERACTIONS_PATH = DATA_ROOT / "raw/yambda_interactions.parquet"
ITEM_EMBEDDINGS_PATH = DATA_ROOT / "raw/yambda_item_embeddings.parquet"
REPORT_DIR = REPO_ROOT / "reports/profiles"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

assert INTERACTIONS_PATH.exists(), INTERACTIONS_PATH

interactions = pl.read_parquet(INTERACTIONS_PATH)
item_embeddings = pl.read_parquet(ITEM_EMBEDDINGS_PATH) if ITEM_EMBEDDINGS_PATH.exists() else None

print(INTERACTIONS_PATH)
print(f"item embeddings present: {ITEM_EMBEDDINGS_PATH.exists()}")
print(f"ydata_profiling available: {ProfileReport is not None}")


/Users/yao/projects/ml-playground/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


../data/raw/yambda_interactions.parquet
item embeddings present: False
ydata_profiling available: True


In [11]:
interactions.head(10)

user_id,item_id,timestamp,target,event_type
i64,i64,i64,i64,str
1,10,1,1,"""listen"""
1,11,2,1,"""listen"""
1,12,3,1,"""listen"""
2,10,1,1,"""listen"""
2,13,2,1,"""listen"""
2,14,3,1,"""listen"""
3,11,1,1,"""listen"""
3,13,2,1,"""listen"""
3,15,3,1,"""listen"""


## Core summary

In [12]:
summary = interactions.select(
    pl.len().alias("rows"),
    pl.n_unique("user_id").alias("unique_users"),
    pl.n_unique("item_id").alias("unique_items"),
    pl.col("timestamp").min().alias("min_timestamp"),
    pl.col("timestamp").max().alias("max_timestamp"),
    pl.col("target").mean().alias("mean_target"),
    pl.n_unique("event_type").alias("event_type_count"),
    ((pl.len()) / (pl.n_unique("user_id") * pl.n_unique("item_id"))).alias("observed_matrix_density")
)

summary

rows,unique_users,unique_items,min_timestamp,max_timestamp,mean_target,event_type_count,observed_matrix_density
u32,u32,u32,i64,i64,f64,u32,f64
9,3,6,1,3,1.0,1,0.5


## Schema, nulls, and duplicates

In [13]:
interaction_schema = pl.DataFrame({
    "column": interactions.columns,
    "dtype": [str(dtype) for dtype in interactions.dtypes],
    "null_count": interactions.null_count().row(0),
})

duplicate_summary = pl.DataFrame({
    "metric": [
        "duplicate_interaction_rows",
        "duplicate_user_item_timestamp_rows",
    ],
    "value": [
        interactions.height - interactions.unique().height,
        interactions.height - interactions.unique(subset=["user_id", "item_id", "timestamp"]).height,
    ],
})

interaction_schema, duplicate_summary

(shape: (5, 3)
 ┌────────────┬────────┬────────────┐
 │ column     ┆ dtype  ┆ null_count │
 │ ---        ┆ ---    ┆ ---        │
 │ str        ┆ str    ┆ i64        │
 ╞════════════╪════════╪════════════╡
 │ user_id    ┆ Int64  ┆ 0          │
 │ item_id    ┆ Int64  ┆ 0          │
 │ timestamp  ┆ Int64  ┆ 0          │
 │ target     ┆ Int64  ┆ 0          │
 │ event_type ┆ String ┆ 0          │
 └────────────┴────────┴────────────┘,
 shape: (2, 2)
 ┌─────────────────────────────────┬───────┐
 │ metric                          ┆ value │
 │ ---                             ┆ ---   │
 │ str                             ┆ i64   │
 ╞═════════════════════════════════╪═══════╡
 │ duplicate_interaction_rows      ┆ 0     │
 │ duplicate_user_item_timestamp_… ┆ 0     │
 └─────────────────────────────────┴───────┘)

## Event and target distribution

In [14]:
event_dist = interactions.group_by("event_type").len().sort("len", descending=True)
target_dist = interactions.group_by("target").len().sort("target")

event_dist, target_dist

(shape: (1, 2)
 ┌────────────┬─────┐
 │ event_type ┆ len │
 │ ---        ┆ --- │
 │ str        ┆ u32 │
 ╞════════════╪═════╡
 │ listen     ┆ 9   │
 └────────────┴─────┘,
 shape: (1, 2)
 ┌────────┬─────┐
 │ target ┆ len │
 │ ---    ┆ --- │
 │ i64    ┆ u32 │
 ╞════════╪═════╡
 │ 1      ┆ 9   │
 └────────┴─────┘)

## User and item activity

In [15]:
user_activity = interactions.group_by("user_id").len().sort("len", descending=True)
item_activity = interactions.group_by("item_id").len().sort("len", descending=True)

activity_profile = pl.DataFrame({
    "metric": [
        "user_interactions_min",
        "user_interactions_p50",
        "user_interactions_p90",
        "user_interactions_max",
        "item_interactions_min",
        "item_interactions_p50",
        "item_interactions_p90",
        "item_interactions_max",
    ],
    "value": [
        float(user_activity["len"].min()),
        float(user_activity["len"].quantile(0.50)),
        float(user_activity["len"].quantile(0.90)),
        float(user_activity["len"].max()),
        float(item_activity["len"].min()),
        float(item_activity["len"].quantile(0.50)),
        float(item_activity["len"].quantile(0.90)),
        float(item_activity["len"].max()),
    ],
})

user_activity, item_activity, activity_profile

TypeError: unexpected value while building Series of type Int64; found value of type Float64: 3.0

Hint: Try setting `strict=False` to allow passing data with mixed types.

## Temporal profile

In [ ]:
timestamp_counts = interactions.group_by("timestamp").len().sort("timestamp")
timestamp_counts

In [ ]:
if plt is None:
    print("matplotlib is not installed; skipping chart")
else:
    pdf = timestamp_counts.to_pandas()
    ax = pdf.plot.bar(x="timestamp", y="len", legend=False, figsize=(7, 4), color="#3b7a57")
    ax.set_title("Interactions per timestamp step")
    ax.set_xlabel("timestamp")
    ax.set_ylabel("count")
    plt.tight_layout()


## Optional item embeddings check

In [ ]:
if item_embeddings is None:
    print("No item embeddings parquet found at ../data/raw/yambda_item_embeddings.parquet")
else:
    embedding_schema = pl.DataFrame({
        "column": item_embeddings.columns,
        "dtype": [str(dtype) for dtype in item_embeddings.dtypes],
        "null_count": item_embeddings.null_count().row(0),
    })
    item_embeddings.head(5), embedding_schema

## Optional ydata-profiling report

Install first if needed:

```bash
uv add ydata-profiling
```


In [ ]:
if ProfileReport is None:
    print("ydata-profiling is not installed. Run: uv add ydata-profiling")
else:
    report = ProfileReport(interactions.to_pandas(), title="Yambda Profile", minimal=True)
    report_path = REPORT_DIR / "yambda_profile_report.html"
    report.to_file(report_path)
    print(report_path)
    report

## Quick observations template

Use this section after running the notebook:

- The current local Yambda parquet is a tiny example-sized interaction table, not a production-scale benchmark snapshot.
- The event stream is structurally simple and useful for smoke tests of sequence and retrieval preparation.
- If you want a fuller Yambda profile, the next missing artifact is `yambda_item_embeddings.parquet`.
